# CafChem GPT — Mini foundation (1 block) training on Colab (GPU)

This notebook trains the **mini** 1-block foundation used by the laptop-friendly
workbench fine-tune (`workbench/`). It is the same architecture and dataset as
the 2-block `data/GPT_ZN305_pytorch.pt`, but with a single transformer block —
roughly half the parameters (~0.7M), so the 1+1-block fine-tune that sits on top
of it runs comfortably on a small Mac.

It clones the `SMILES_GPT` repo, rebuilds the `ZN305K_tokenized.npz` cache
(gitignored, so rebuilt once on Colab), trains a fresh 1-block model
(`make_gpt(1, ...)`) on `data/ZN305K_smiles.csv`, saves `data/GPT_ZN305_mini.pt`,
and lets you download it back to commit alongside the 2-block foundation.

Recipe matches the 2-block foundation: `key_dim=256`, `batch 128`, constant
`LR=1e-3`, unmasked loss. Cells run top-to-bottom.

## 1. Clone the repo

The repo is public, so a plain `git clone` is all that's needed.

In [ ]:
#@title Clone SMILES_GPT
import os, shutil
REPO_DIR = "SMILES_GPT"  #@param {type:"string"}

# Wipe a stale clone so re-runs are clean.
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone https://github.com/MauricioCafiero/SMILES_GPT.git {REPO_DIR}

# Work from the repo root so all relative paths (data/, code/) resolve.
os.chdir(REPO_DIR)
print("Cwd:", os.getcwd())
print("Files:", os.listdir("."))

## 2. Install dependencies

Colab already ships `torch` (CUDA build), `pandas`, `numpy`, `Pillow`.
We only need to add `rdkit` (imported by `CafChemGPT.py`).

In [ ]:
#@title Install rdkit
# Colab already ships torch (CUDA), pandas, numpy, Pillow. Only rdkit is needed.
!pip install -q rdkit
import rdkit
print("rdkit", rdkit.__version__)

## 3. Verify the GPU

`get_device` picks CUDA on Colab. Confirm a T4 (or better) is attached. We also
note whether a `data/GPT_ZN305_mini.pt` already exists (this run overwrites it).

In [ ]:
import torch, os
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")
else:
    print("[!] No GPU — go to Runtime > Change runtime type > T4 GPU.")

ckpt = "data/GPT_ZN305_mini.pt"
print(f"\nMini checkpoint {ckpt}: "
      f"{'present (will be overwritten)' if os.path.exists(ckpt) else 'not yet present'}")

## 4. (Re)build the tokenized dataset cache

The `ZN305K_tokenized.npz` cache (402 MB) is gitignored and is **not** in the
repo, so we rebuild it once from `data/ZN305K_smiles.csv`. This takes a few
minutes. (Same cache the 2-block foundation notebook uses.)

In [ ]:
import os, sys, numpy as np
sys.path.insert(0, "code")

from CafChemGPT import make_datasets, get_device
print("Device:", get_device())

# This cache stores token-id arrays from the (fixed, 2026-07-21) tokenizer.
# Run from cell 1 (fresh clone) so it rebuilds, or delete the .npz to force it.
cache = "data/ZN305K_tokenized.npz"
if os.path.exists(cache):
    z = np.load(cache)
    print(f"Cache already present: fx {z['fx'].shape}, fy {z['fy'].shape}, "
          f"VOCAB {int(z['VOCAB_SIZE'])}, max_len {int(z['max_length'])}")
else:
    print("Rebuilding tokenized cache from data/ZN305K_smiles.csv ...")
    fx, fy, VOCAB_SIZE, tokenizer, max_length = make_datasets(
        "data/ZN305K_smiles.csv", "SMILES")
    np.savez(cache, fx=fx, fy=fy, VOCAB_SIZE=VOCAB_SIZE, max_length=max_length)
    print(f"Saved {cache}: fx {fx.shape}, fy {fy.shape}, "
          f"VOCAB {VOCAB_SIZE}, max_len {max_length}")

## 5. Train the mini foundation (1 block) from scratch

Builds a fresh 1-block GPT (`make_gpt(1, max_length, VOCAB_SIZE)`) — the same
`key_dim=256` attention as the 2-block foundation, just a single transformer
block — trains `EXTRA_EPOCHS` at constant `LR=1e-3` (matching the TF recipe),
and saves to `data/GPT_ZN305_mini.pt`. `train_gpt` auto-enables bf16 autocast on
CUDA, so one block is fast on an A100 and 150 epochs is cheap. After training,
run cell 6 to download the `.pt` and commit it so `workbench/finetune_gpt.py` can
load it locally.

In [ ]:
#@title Train the mini foundation (1 block) from scratch
import os, sys, numpy as np
sys.path.insert(0, "code")

EXTRA_EPOCHS = 150  #@param {type:"integer"}
BATCH_SIZE   = 128  #@param {type:"integer"}
LR           = 1e-3 #@param {type:"number"}

from CafChemGPT import make_gpt, train_gpt, save_gpt, get_device

device = get_device()
cache = "data/ZN305K_tokenized.npz"
z = np.load(cache)
fx, fy = z["fx"], z["fy"]
VOCAB_SIZE, max_length = int(z["VOCAB_SIZE"]), int(z["max_length"])
print(f"fx {fx.shape} | fy {fy.shape} | VOCAB {VOCAB_SIZE} | max_len {max_length}")
print("Device:", device)

# 1-block mini foundation: same key_dim=256 architecture as the 2-block one,
# just a single transformer block (~0.7M params). batch 128 + constant LR 1e-3
# match the TF recipe; train_gpt auto-enables bf16 autocast on CUDA.
MINI_FILE = "data/GPT_ZN305_mini"  # .pt appended by save_gpt
gpt = make_gpt(1, max_length, VOCAB_SIZE)
gpt = train_gpt(gpt, fx, fy, epochs=EXTRA_EPOCHS, batch_size=BATCH_SIZE, lr=LR)
save_gpt(gpt, MINI_FILE)
print(f"\nMini foundation saved -> {MINI_FILE}.pt")

## 6. Download the `.pt` back to your Mac

Run this cell to download `data/GPT_ZN305_mini.pt` to your browser, drop it into
the local repo's `data/` folder, and commit it so `workbench/` can load it. ~2–3 MB.

In [ ]:
from google.colab import files
import os

pt_path = "data/GPT_ZN305_mini.pt"
print(f"Downloading {pt_path} ({round(os.path.getsize(pt_path)/1e6,1)} MB) ...")
files.download(pt_path)

## 7. (Optional) Quick generation sanity check

Generate a few molecules from the freshly trained mini foundation to confirm it
produces valid SMILES. The more epochs trained, the fewer invalid molecules you
should see.

In [ ]:
import sys; sys.path.insert(0, "code")
from CafChemGPT import load_gpt, make_prompts, gen_mols
from smiles_tokenizer import SmilesTokenizer
from IPython.display import display
from rdkit import Chem

tok = SmilesTokenizer(vocab_file="data/vocab_305K.txt")
VOCAB = tok.vocab_size()
model = load_gpt("data/GPT_ZN305_mini", 1, 166, VOCAB)  # 1 block

prompts = make_prompts(8, 2)
pic, smiles = gen_mols(prompts, use_ramp=True, model=model, tokenizer=tok,
                       TEMP=1.5, VOCAB_SIZE=VOCAB)

# Validity rate (fraction that RDKit can parse).
valid = [s for s in smiles if Chem.MolFromSmiles(s) is not None]
print(f"{len(valid)}/{len(smiles)} valid SMILES. Samples:", smiles[:5])

# Robust display: save if possible, otherwise just display directly.
if hasattr(pic, "save"):
    pic.save("mini_generated_molecules.png")
    from IPython.display import Image
    display(Image("mini_generated_molecules.png"))
else:
    display(pic)